# 03 - No-cloning, teleportation và superdense coding

Ba bài trước dùng entanglement rất ít. Bài này dùng nó làm nhân vật chính, qua hai giao thức
đối xứng nhau một cách đẹp đẽ:

| | Gửi gì | Qua gì | Cần thêm |
|---|---|---|---|
| **Teleportation** | 1 qubit | 2 bit cổ điển | 1 cặp entangled |
| **Superdense coding** | 2 bit cổ điển | 1 qubit | 1 cặp entangled |

Và để hiểu tại sao teleportation phải tốn công như vậy, trước tiên cần biết một điều không làm được:
**không thể copy một qubit**.

Cần trước: `00_fundamental/02_cong_nhieu_qubit.ipynb` (trạng thái Bell).
Bài này cũng giới thiệu **mid-circuit measurement** và **điều khiển cổ điển** (`if_test`).

In [ ]:
import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit

print("qiskit:", qiskit.__version__)

sampler = SamplerV2()


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def do(qc, shots=1024, reg="c"):
    """Chạy mạch và trả về counts từ thanh ghi cổ điển tên `reg`."""
    ket_qua = sampler.run([qc], shots=shots).result()[0]
    return getattr(ket_qua.data, reg).get_counts()

## 1. Định lý no-cloning

> Không tồn tại mạch lượng tử nào copy được một trạng thái tùy ý:
> $\lvert \psi\rangle\lvert 0\rangle \mapsto \lvert \psi\rangle\lvert \psi\rangle$ với **mọi** $\psi$.

Chứng minh ngắn: cổng lượng tử là **tuyến tính**. Nếu một mạch copy được $\lvert 0\rangle$ và
$\lvert 1\rangle$, thì tác dụng của nó lên $\lvert +\rangle = \frac{1}{\sqrt2}(\lvert 0\rangle+\lvert 1\rangle)$
**buộc phải** là

$$\tfrac{1}{\sqrt2}\big(\lvert 00\rangle + \lvert 11\rangle\big) \quad(\text{trạng thái Bell})$$

trong khi bản copy thật sự phải là

$$\lvert +\rangle\lvert +\rangle = \tfrac12\big(\lvert 00\rangle+\lvert 01\rangle+\lvert 10\rangle+\lvert 11\rangle\big)$$

Hai cái này khác nhau, nên máy copy đó không tồn tại. `CX` chính là "máy copy" cho computational basis -
và dưới đây là bằng chứng nó trượt ngay khi gặp superposition.

In [ ]:
print("CX copy được các ket cơ sở:")
for bit in ["0", "1"]:
    qc = QuantumCircuit(2)
    if bit == "1":
        qc.x(0)
    qc.cx(0, 1)
    print(f"  |{bit}>|0> -> {ket(Statevector(qc))}")

print("\nNhưng với |+> thì không:")
thu_copy = QuantumCircuit(2)
thu_copy.h(0)
thu_copy.cx(0, 1)
print("  CX cho ra         :", ket(Statevector(thu_copy)))

copy_that = QuantumCircuit(2)
copy_that.h(0)
copy_that.h(1)
print("  copy thật phải là :", ket(Statevector(copy_that)))
print("\n  -> CX tạo ra trạng thái ENTANGLED, không phải hai bản sao độc lập.")

Hai hệ quả:

- **Không thể "gửi qubit" bằng cách copy rồi giữ lại bản gốc.** Muốn chuyển trạng thái đi thì
  bản gốc buộc phải bị phá hủy - và đúng là teleportation làm vậy.
- Không thể khuếch đại tín hiệu lượng tử như khuếch đại điện, nên "repeater" lượng tử là bài toán khó.

## 2. Teleportation

Alice có qubit $q_0$ ở trạng thái $\lvert \psi\rangle$ **không biết trước**. Cô muốn Bob có đúng
trạng thái đó trên qubit $q_2$ của anh ta. Họ chia sẵn một cặp entangled ($q_1$ của Alice, $q_2$ của Bob).

Các bước:

1. Alice "trộn" $\lvert\psi\rangle$ với phần entangled của mình: `cx(0, 1)` rồi `h(0)`
2. Alice **đo** cả hai qubit của mình -> được 2 bit cổ điển
3. Alice gọi điện thoại nói 2 bit đó cho Bob (đường truyền **cổ điển**, tối đa bằng tốc độ ánh sáng)
4. Bob sửa qubit của mình theo 2 bit nhận được: bit thứ nhất -> áp $X$, bit thứ hai -> áp $Z$

Sau bước 4, $q_2$ ở đúng trạng thái $\lvert\psi\rangle$, còn qubit gốc của Alice đã collapse
(không còn $\lvert\psi\rangle$ nữa - đúng như no-cloning đòi hỏi).

Bước 4 cần thứ mới: **mid-circuit measurement** rồi **dùng kết quả cổ điển để điều khiển cổng**. Trong
Qiskit 2.x viết bằng `with qc.if_test((bit, 1)):`.

In [ ]:
def mach_teleport(prep):
    """prep = mạch 1 qubit tạo ra trạng thái cần gửi."""
    q = QuantumRegister(3, "q")
    m0 = ClassicalRegister(1, "m0")      # bit đo từ q1
    m1 = ClassicalRegister(1, "m1")      # bit đo từ q0
    kt = ClassicalRegister(1, "kiemtra")
    qc = QuantumCircuit(q, m0, m1, kt)

    qc.compose(prep, qubits=[0], inplace=True)   # Alice có |psi> trên q0
    qc.barrier()

    qc.h(1)                                      # cặp entangled q1 (Alice) - q2 (Bob)
    qc.cx(1, 2)
    qc.barrier()

    qc.cx(0, 1)                                  # Alice trộn rồi đo
    qc.h(0)
    qc.measure(1, m0[0])
    qc.measure(0, m1[0])
    qc.barrier()

    with qc.if_test((m0[0], 1)):                 # Bob sửa theo 2 bit nhận được
        qc.x(2)
    with qc.if_test((m1[0], 1)):
        qc.z(2)
    qc.barrier()

    qc.compose(prep.inverse(), qubits=[2], inplace=True)   # "gỡ" lại để kiểm tra
    qc.measure(2, kt[0])
    return qc


prep = QuantumCircuit(1, name="psi")
prep.ry(1.0, 0)
prep.rz(0.7, 0)

qc = mach_teleport(prep)
print(qc.draw(fold=120))

### Làm sao kiểm tra là đã đúng?

Không thể gọi `Statevector(qc)` vì mạch có lệnh đo giữa chừng (thử đi, Qiskit sẽ báo lỗi).
Thủ đoạn chuẩn: **áp ngược lại mạch chuẩn bị** (`prep.inverse()`) lên qubit của Bob rồi đo.

Nếu teleport đúng, $q_2$ đang ở $\lvert\psi\rangle$, nên $\text{prep}^\dagger\lvert\psi\rangle = \lvert 0\rangle$
và ta phải đo được `'0'` **100% số lần**. Bất kỳ bit `'1'` nào cũng là bằng chứng sai.

In [ ]:
for ry, rz in [(1.0, 0.7), (0.3, 2.1), (np.pi / 2, 0.0), (2.5, 1.3)]:
    p = QuantumCircuit(1)
    p.ry(ry, 0)
    p.rz(rz, 0)
    c = do(mach_teleport(p), shots=2048, reg="kiemtra")
    print(f"  psi = ry({ry:.2f}) rz({rz:.2f}) -> kiemtra {c}  "
          f"{'OK' if list(c) == ['0'] else 'SAI'}")

In [ ]:
# Nếu BỎ bước sửa của Bob thì sao? -> ra ngẫu nhiên 50/50
q = QuantumRegister(3, "q")
m = ClassicalRegister(2, "m")
kt = ClassicalRegister(1, "kiemtra")
khong_sua = QuantumCircuit(q, m, kt)
khong_sua.compose(prep, qubits=[0], inplace=True)
khong_sua.h(1)
khong_sua.cx(1, 2)
khong_sua.cx(0, 1)
khong_sua.h(0)
khong_sua.measure(1, m[0])
khong_sua.measure(0, m[1])
khong_sua.compose(prep.inverse(), qubits=[2], inplace=True)
khong_sua.measure(2, kt[0])

print("không sửa ->", do(khong_sua, shots=2048, reg="kiemtra"))
print("có sửa    ->", do(mach_teleport(prep), shots=2048, reg="kiemtra"))
print("\n-> 2 bit cổ điển là thành phần BẮT BUỘC, không phải thủ tục rườm rà.")

### Vì sao đây không phải truyền tin nhanh hơn ánh sáng

Trước khi nhận 2 bit của Alice, qubit của Bob ở trạng thái **hoàn toàn ngẫu nhiên** - ô ngay
trước chính là bằng chứng: bỏ bước sửa thì kết quả là 50/50, không mang thông tin gì. Thông tin
chỉ hoàn chỉnh khi 2 bit cổ điển tới, mà chúng đi bằng đường truyền thông thường.

Cũng lưu ý: "teleportation" không chuyển **vật chất**, chỉ chuyển **trạng thái**. Và bản gốc bị
phá hủy - nên gọi là "di chuyển" thì đúng hơn là "nhân bản".

## 3. Superdense coding: chiều ngược lại

Alice muốn gửi **2 bit cổ điển** cho Bob nhưng chỉ được gửi **1 qubit**. Điều này nghe như
vi phạm giới hạn thông tin - và đúng là vậy nếu không có entanglement sẵn.

Với một cặp entangled chia trước, Alice chỉ cần tác động lên qubit của **chính cô** rồi gửi nó đi:

| 2 bit muốn gửi | Alice áp |
|---|---|
| `00` | không gì |
| `01` | $X$ |
| `10` | $Z$ |
| `11` | $Z$ rồi $X$ |

Bốn phép này biến cặp entangled thành 4 trạng thái Bell khác nhau - mà 4 trạng thái Bell **trực giao**
nên Bob phân biệt được chắc chắn: anh ta chỉ cần đảo ngược mạch tạo Bell (`cx` rồi `h`) và đo.

In [ ]:
def superdense(tin_nhan):
    """tin_nhan = 2 ký tự 'b1b0'. Trả về mạch hoàn chỉnh."""
    b1, b0 = tin_nhan[0], tin_nhan[1]
    qc = QuantumCircuit(2, 2)

    qc.h(0)              # cặp entangled: q0 của Alice, q1 của Bob
    qc.cx(0, 1)
    qc.barrier()

    if b0 == "1":        # Alice mã hóa, chỉ động vào q0
        qc.x(0)
    if b1 == "1":
        qc.z(0)
    qc.barrier()
    # ... Alice gửi q0 cho Bob ...

    qc.cx(0, 1)          # Bob giải mã
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc


print(superdense("10").draw())

Một chỗ dễ nhầm khi đọc kết quả: chuỗi counts của Qiskit là `c1c0` (**little-endian**, bit của
qubit 0 nằm bên phải), trong khi tin nhắn ta viết là `b1b0` với $b_0$ là bit điều khiển $X$.
Hóa ra $b_0$ lại hiện ra ở **c1**, nên chuỗi thô bị **đảo ngược** so với tin nhắn. Phải đảo lại
khi đọc - đây không phải lỗi thuật toán, chỉ là quy ước thứ tự bit.

In [ ]:
print("Mapping thô: (có X?, có Z?) -> chuỗi counts")
for coX in [False, True]:
    for coZ in [False, True]:
        qc = QuantumCircuit(2, 2)
        qc.h(0)
        qc.cx(0, 1)
        if coX:
            qc.x(0)
        if coZ:
            qc.z(0)
        qc.cx(0, 1)
        qc.h(0)
        qc.measure([0, 1], [0, 1])
        tho = list(do(qc, shots=256))[0]
        print(f"  X={coX!s:5s} Z={coZ!s:5s} -> '{tho}'")

print("\nGửi đủ 4 tin nhắn (đảo chuỗi khi đọc):")
for tin in ["00", "01", "10", "11"]:
    c = do(superdense(tin), shots=512)
    tho = list(c)[0]
    nhan = tho[::-1]
    print(f"  gửi {tin} -> counts {c} -> đảo lại '{nhan}'  "
          f"{'OK' if nhan == tin else 'SAI'}")

Lưu ý để không hiểu sai: **tổng số qubit di chuyển vẫn là 2** - một qubit đi trước (lúc chia cặp entangled)
và một qubit đi sau (lúc Alice gửi). Superdense coding không phá vỡ giới hạn Holevo
("$n$ qubit chỉ tải được $n$ bit"); nó chỉ cho phép **dồn** một nửa chi phí truyền thông sang
lúc rảnh rỗi, trước khi biết mình sẽ gửi tin gì.

## Tóm tắt

| Khái niệm | Nội dung |
|---|---|
| No-cloning | không copy được trạng thái tùy ý; hệ quả của tính tuyến tính |
| Teleportation | 1 qubit = 1 cặp entangled + 2 bit cổ điển; bản gốc bị phá hủy |
| Mid-circuit measurement | `qc.measure(...)` rồi `with qc.if_test((bit, 1)):` để điều khiển cổ điển |
| Kiểm tra teleport | áp `prep.inverse()` rồi đo, phải ra `'0'` 100% |
| Superdense | 2 bit cổ điển = 1 cặp entangled + 1 qubit; 4 phép $I, X, Z, ZX$ -> 4 trạng thái Bell |
| Không vi phạm vật lý | teleport cần bit cổ điển (không nhanh hơn ánh sáng); superdense vẫn tốn 2 qubit |

### Thử tự làm

1. Trong `mach_teleport`, đổi thứ tự hai bước sửa của Bob ($Z$ trước rồi $X$). Kết quả có đổi không?
   Giải thích bằng cách xét $XZ$ và $ZX$ khác nhau dấu thế nào.
2. Thử gọi `Statevector(mach_teleport(prep))` để thấy lỗi, và đọc kỹ thông báo lỗi.
3. Hiện thực `superdense` cho trường hợp Alice áp $Y$ thay vì $ZX$. Kết quả đọc được là gì,
   và vì sao (gợi ý: $Y = iXZ$)?

## Học gì tiếp

- **QFT** (quantum Fourier transform) và **phase estimation** - cửa ngõ vào Shor; QFT thực chất
  là "kickback" làm một cách hệ thống trên nhiều qubit.
- **VQE / variational circuit** - hướng được dùng nhiều nhất trên máy thực tế hiện nay, cần
  `EstimatorV2` và tối ưu hóa cổ điển.
- **Mô phỏng nhiễu** (`qiskit_aer.noise`) - xem mạch của bạn hỏng thế nào trên máy thật,
  và vì sao `depth` và số `cx` lại quan trọng.